# 第2幕『方向を知る』 — 拒否はどう表現されているか（論文読解）

> **シリーズの物語** 「小さなモデルを、三つの角度から解剖する」
>
> 同じ小型モデル（Qwen2.5-0.5B など）を主役にして、3冊を通して一本の筋で学びます。
>
> 1. **第1幕『縮める』** — 量子化。重みの精度を落とすと、サイズ・速度・賢さがどう動くかを測る。
> 2. **第2幕『方向を知る』** — abliteration 論文の読解。「拒否は1つの方向に乗っている」という主張を、概念として理解する（コードで外す作業はしない）。
> 3. **第3幕『中を覗く』** — 解釈性。第2幕の土台である「内部表現を読む」技術を、安全な題材（事実の想起）で自分の手で動かす。
>
> 縮めて速くし、その中で何が起きているかを知り、最後に内部を直接観察する——という流れです。


> **このノートの立場** これは **概念を理解するための読解ノート**です。
> 安全対策を実際に取り除くコードは含みません（それは目的を問わず扱いません）。
> ここで学ぶのは「モデルの中で振る舞いがどう表現されうるか」という**解釈性の考え方**で、
> その技術そのものは第3幕で**安全な題材**を使って自分の手で動かします。


## 読む論文

**Arditi, Obeso, Syed, Paleka, Panickssery, Gurnee, Nanda (2024)**
*"Refusal in Language Models Is Mediated by a Single Direction"* (NeurIPS 2024)
arXiv:2406.11717

主張を一言で: **「チャットモデルが指示を"断る"という振る舞いは、残差ストリーム上のたった1つの方向にほぼ集約されている」**。


## 前提: 残差ストリーム（residual stream）という考え方

Transformer は各トークンについて「隠れ状態ベクトル」を持ち、層を通るたびに
Attention と MLP が**その上に足し算**をしていきます。この積み上がっていく共有ベクトル空間を
**残差ストリーム**と呼びます。

重要なのは、**概念や振る舞いが、この空間の"方向（ベクトル）"として表現されることがある**という見方です。
「肯定的／否定的」「英語／フランス語」のような軸が、特定方向への射影として現れる、という観察が
解釈性研究で積み重なってきました。論文はこれを「拒否」という振る舞いに当てはめます。

> ⚠️ これは**経験的な観察**であって、全モデル・全振る舞いで厳密に成り立つ保証はありません。
> この但し書きは最後にもう一度強調します。


## 論文のロジック（4ステップ）

絵で描くとこうなります。コードではなく**考え方の流れ**として読んでください。


### ① 対比プロンプトを2群用意する
- **有害な指示**の集合 H（断られやすい）
- **無害な指示**の集合 B（断られない）

両方をモデルに通し、ある層・ある位置の**残差ストリームのベクトル**を集める。

### ② 平均の差を取る → 「方向」を推定する
$$ d = \overline{h\in H} - \overline{b\in B} $$

「断る時の内部状態」と「断らない時の内部状態」の**平均の差ベクトル** $d$ が、
拒否に関係する方向の候補になる（difference-in-means という古典的な手法）。

### ③ その方向が"効いている"か検証する（因果）
- **足す（activation addition）**: 無害プロンプトの実行中に $d$ 方向を足すと、**断り始める**。
- **消す（projection を引く）**: 実行中に $d$ 成分を取り除くと、**断りにくくなる**。

相関だけでなく「いじると振る舞いが変わる」= **因果的**に効いている、と示すのがこの論文の肝。

### ④ 「単一方向」の強さ
多くの指示・多くのモデルで、**ほぼ1本の方向**でこの操作が成立した——というのが結論。


## 図解: 残差空間の中の「拒否方向」（概念図）

コードは**描画のためだけ**で、モデルは使いません。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(0)
# 概念図: 2次元に落とした"内部状態"。無害(青)と有害(赤)が1方向に分かれる様子
d = np.array([2.2, 0.6]); d = d/np.linalg.norm(d)   # 「拒否方向」(あくまで作図用の仮想データ)
benign  = rng.normal([-1.0,0.0],0.5,(60,2))
harmful = rng.normal([ 1.4,0.0],0.5,(60,2)) + d*1.3

fig, ax = plt.subplots(figsize=(6,5))
ax.scatter(*benign.T,  c="tab:blue", label="無害な指示（断らない）", alpha=.7)
ax.scatter(*harmful.T, c="tab:red",  label="有害な指示（断る）",   alpha=.7)
m = (benign.mean(0)+harmful.mean(0))/2
ax.annotate("", xy=m+d*2, xytext=m-d*2,
            arrowprops=dict(arrowstyle="->", lw=2, color="k"))
ax.text(*(m+d*2.1), "推定された『方向 d』", fontsize=11)
ax.set_title("概念図: 断る/断らない が1方向で分かれる"); ax.legend(); ax.set_aspect("equal")
plt.tight_layout(); plt.show()
print("※ これはダミーデータの作図です。実モデルの数値ではありません。")


## なぜこれが「解釈性の勝利」として面白いのか

- 巨大で不透明に見えるモデルの振る舞いが、**人間が名前を付けられる1本の軸**に対応していた。
- しかも**因果的に操作できる**ので、「ただの相関では？」という批判に答えられる。
- 同じ difference-in-means → add/ablate の枠組みは、**拒否に限らず**
  「フォーマルさ」「言語」「感情」など**他の多くの軸**の研究に使われている汎用ツール。

第3幕では、この「差ベクトルを作って足し引きして因果を見る」という**まさに同じ道具**を、
**事実の想起**という無害な題材で動かします。


## 限界と但し書き（ここが一番大事）

論文自身・その後の議論で指摘されている点:

1. **「単一」は近似**。拒否は主に1方向だが、完全に1次元とは限らない。層・モデルで分布する残差がある。
2. **モデル依存**。測り方（どの層・どの位置・どのプロンプト集合）で方向は変わる。
3. **安全性の含意**。この知見は「安全対策が意外と脆い」ことも示す。だからこそ、
   - **防御側**（安全対策をより頑健にする、どこが壊れやすいか知る）に価値がある一方、
   - 攻撃的な複製はこのシリーズでは扱わない。
4. **評価の難しさ**。「断らなくなった」ことの測定自体にノイズが多い。

> まとめ: *振る舞いはしばしば低次元の方向に乗る* という**観察**は強力だが、
> 普遍法則ではなく**経験則**。鵜呑みにせず、常に「どの設定で測ったか」を問うこと。


## まとめ（第2幕）

- 拒否という複雑そうな振る舞いが、残差ストリームの**ほぼ1方向**に対応する、という経験的発見。
- 手法は **difference-in-means で方向を推定 → 足す/射影で消す → 因果を確認**。
- これは解釈性の**汎用ツール**であり、対象は拒否に限らない。
- ただし「単一方向」は**近似**で、設定依存。限界を忘れないこと。

> **次の幕へ** — 「内部状態をベクトルとして読む」「方向を足し引きして因果を見る」。
> この道具を、いよいよ**自分の手で**動かします。題材は安全な『事実の想起』です。
